#### Preparation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.models import swin_t, Swin_T_Weights
from torch.utils.data import DataLoader
from PIL import Image
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm

In [2]:
DATA_DIR = r"C:\Users\yugil\Downloads\Cassava_Leaf_Datasets\Cassava_Leaf_Datasets\Classification"
WEIGHTS_DIR = r'../weights'

BATCH_SIZE = 32
FREEZE_EPOCHS = 8
FINETUNE_EPOCHS = 20

LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
weights = Swin_T_Weights.DEFAULT

mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=mean,
        std=std
    )
])

In [4]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "val"),
    transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['Healthy', 'Spider Mites', 'leaf blight', 'leaf spot']


In [5]:
model = swin_t(weights=weights)

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to C:\Users\yugil/.cache\torch\hub\checkpoints\swin_t-704ceda3.pth
100%|██████████| 108M/108M [00:02<00:00, 50.6MB/s] 


In [6]:
for param in model.parameters():
    param.requires_grad = False

In [7]:
# Replace Classification Head
in_features = model.head.in_features

model.head = nn.Linear(in_features, num_classes)

In [8]:
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.head.parameters(),
    lr=LR
)

In [9]:
def train_one_epoch(loader, epoch, epochs):
    model.train()
    
    train_loss = 0
    train_preds = []
    train_labels = []

    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch+1}/{epochs}"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)

        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(labels.cpu().numpy())

        train_acc = accuracy_score(train_labels, train_preds)


    return train_loss, train_acc


In [10]:
def validate(loader):
    model.eval()
    
    val_loss = 0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating "):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = outputs.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
        
        val_acc = accuracy_score(val_labels, val_preds)

        precision, recall, f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted'
        )   

    return precision, recall, f1, val_loss, val_acc

#### Training 1 (Freeze-Backbone)

In [11]:
for epoch in range(FREEZE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FREEZE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FREEZE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")

Validating : 100%|██████████| 8/8 [00:31<00:00,  3.91s/it]



Epoch [1/8]
Train Loss: 40.9360 | Train Acc: 0.3844
Val Loss: 9.9105 | Val Acc: 0.5500
Precision: 0.6139 | Recall: 0.5500 | F1: 0.5411




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.77s/it]



Epoch [2/8]
Train Loss: 36.3173 | Train Acc: 0.5792
Val Loss: 8.9825 | Val Acc: 0.6958
Precision: 0.7276 | Recall: 0.6958 | F1: 0.6882




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.77s/it]


...


Epoch [6/8]
Train Loss: 26.4897 | Train Acc: 0.7771
Val Loss: 6.6655 | Val Acc: 0.7542
Precision: 0.7875 | Recall: 0.7542 | F1: 0.7443




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.81s/it]



Epoch [7/8]
Train Loss: 25.0084 | Train Acc: 0.7750
Val Loss: 6.3272 | Val Acc: 0.7542
Precision: 0.7871 | Recall: 0.7542 | F1: 0.7458




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.76s/it]


Epoch [8/8]
Train Loss: 23.5123 | Train Acc: 0.8229
Val Loss: 6.0440 | Val Acc: 0.7500
Precision: 0.7902 | Recall: 0.7500 | F1: 0.7379




#### Training 2 (Fine-Tuning)

In [12]:
# Unfreeze last block and head for fine-tuning
for param in model.features[-1].parameters():
    param.requires_grad = True

In [13]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

In [14]:
best_val_acc = 0.0

print("\n-----------Starting Fine-tuning Stage-----------\n")

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_one_epoch(train_loader, epoch, FINETUNE_EPOCHS)
    precision, recall, f1, val_loss, val_acc = validate(val_loader)

    print(f"\nEpoch [{epoch+1}/{FINETUNE_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}\n\n")



torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names
        }, os.path.join(WEIGHTS_DIR, "Swint-T.pth"))


-----------Starting Fine-tuning Stage-----------



Validating : 100%|██████████| 8/8 [00:30<00:00,  3.78s/it]



Epoch [1/20]
Train Loss: 20.8579 | Train Acc: 0.8167
Val Loss: 4.7814 | Val Acc: 0.7792
Precision: 0.8029 | Recall: 0.7792 | F1: 0.7747




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.79s/it]



Epoch [2/20]
Train Loss: 16.1834 | Train Acc: 0.8448
Val Loss: 3.9593 | Val Acc: 0.8125
Precision: 0.8226 | Recall: 0.8125 | F1: 0.8104




...


Epoch [18/20]
Train Loss: 5.6553 | Train Acc: 0.9354
Val Loss: 1.7794 | Val Acc: 0.9333
Precision: 0.9340 | Recall: 0.9333 | F1: 0.9335




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.78s/it]



Epoch [19/20]
Train Loss: 5.3236 | Train Acc: 0.9448
Val Loss: 1.7741 | Val Acc: 0.9417
Precision: 0.9430 | Recall: 0.9417 | F1: 0.9420




Validating : 100%|██████████| 8/8 [00:30<00:00,  3.76s/it]


Epoch [20/20]
Train Loss: 5.2125 | Train Acc: 0.9417
Val Loss: 1.7682 | Val Acc: 0.9375
Precision: 0.9384 | Recall: 0.9375 | F1: 0.9378




#### Prediction Sample

In [15]:
checkpoint = torch.load(r"../weights/Swint-T.pth")

model.load_state_dict(checkpoint["model_state_dict"])
class_names = checkpoint["class_names"]

model.eval()

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Example usage:
prediction = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Healthy\Healthy_val_13.jpg")
prediction1 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Blight\leaf blight_val_15.jpg")
prediction2 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Leaf Spot\leaf spot_val_14.jpg")
prediction3 = predict(r"C:\Users\johnp\Desktop\Dataset\Cassava_Leaf_Datasets\Classification\val\Spider Mites\Spider Mites_val_24.jpg")
print(f"Healthy Predicted class: {prediction}")
print(f"Leaf Blight Predicted class: {prediction1}")
print(f"Leaf Spot Predicted class: {prediction2}")
print(f"Spider Mites Predicted class: {prediction3}")

C:\Users\yugil\AppData\Local\Temp\ipykernel_29856\1380619959.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(r"../weights/Swint-T.pth")


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\johnp\\Desktop\\Dataset\\Cassava_Leaf_Datasets\\Classification\\val\\Healthy\\Healthy_val_13.jpg'